# Lab 2: Regular Expressions, Edit Distance, and Subword Unit Tokenization

In this lab, we will cover key topics, including regular expressions, edit distance, and subword unit tokenization using Byte Pair Encoding (BPE). This notebook contains exercises that students will need to complete in the code sections. The necessary files will also be provided with this notebook.


## Exercise 1: Basic Regular Expressions

Write regular expressions for the following languages:
1. The set of all alphabetic strings
2. The set of all lower-case alphabetic strings ending in a b
3. The set of all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b

In [7]:
import re

# 1. The set of all alphabetic strings
regex_1 = r'[A-Za-z]+'


# 2. The set of all lower-case alphabetic strings ending in the letter b
regex_2 = r'[a-z]*b$'

# 3. The set of all strings from the alphabet a,b such that each letter a is immediately preceded by and immediately followed by the letter b
regex_3 = r'(b+(ab)*b*)*'

# Test the regular expressions
test_strings = ["", "Hello", "world", "abc", "b", "bb", "bab", "babab", "abab",
                "babbabbab", "bababab", "abba", "ba", "baab", "ab"]

for i, (regex,question) in enumerate([(regex_1,"all alphabetic strings"), (regex_2,"all lower-case alphabetic strings ending in a b"), (regex_3,"all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b")], 1):
    print(f"\nTesting regex {regex} which searches for {question}:")
    for test_str in test_strings:
        print(f"\"{test_str}\" matches: {bool(re.fullmatch(regex, test_str))}")


Testing regex [A-Za-z]+ which searches for all alphabetic strings:
"" matches: False
"Hello" matches: True
"world" matches: True
"abc" matches: True
"b" matches: True
"bb" matches: True
"bab" matches: True
"babab" matches: True
"abab" matches: True
"babbabbab" matches: True
"bababab" matches: True
"abba" matches: True
"ba" matches: True
"baab" matches: True
"ab" matches: True

Testing regex [a-z]*b$ which searches for all lower-case alphabetic strings ending in a b:
"" matches: False
"Hello" matches: False
"world" matches: False
"abc" matches: False
"b" matches: True
"bb" matches: True
"bab" matches: True
"babab" matches: True
"abab" matches: True
"babbabbab" matches: True
"bababab" matches: True
"abba" matches: False
"ba" matches: False
"baab" matches: True
"ab" matches: True

Testing regex (b+(ab)*b*)* which searches for all strings from the alphabet a,b such that each a is immediately preceded by and immediately followed by a b:
"" matches: True
"Hello" matches: False
"world" match

## Exercise 2: Advanced Regular Expressions

Write regular expressions for the following languages. By "word", we mean an alphabetic string separated from other words by whitespace, any relevant punctuation, line breaks, and so forth.

1. The set of all strings with two consecutive repeated words (e.g., "Humbert Humbert" and "the the" but not "the bug" or "the big bug").
2. All strings that start at the beginning of the line with an integer and that end at the end of the line with a word.
3. All strings that have both the word grotto and the word raven in them (but not, e.g., words like grottos that merely contain the word grotto).
4. Write a pattern that captures the first word of an English sentence along with any ending punctuation if it is concatenated to it.

In [33]:
import re



regex_1 = r"\b([A-Za-z]+)\b[\W_]+\b\1\b"


regex_2 = r"^\d+\s+.*[A-Za-z]+$"


regex_3 = r"((.*\bgrotto\b)(.*\braven\b))|((.*\braven\b)(.*\bgrotto\b))"


regex_4 = r"^\s*[A-Za-z]+[.,!?;:]?"


def run_tests(title, pattern, flags, cases, group=0):
    print(f"\nTesting {title}: {pattern}")
    for text, expected in cases:
        match = re.search(pattern, text, flags)
        found = match.group(group) if match else None
        status = "OK" if bool(match) == expected else "WRONG"
        print(f"  {text!r:55} -> {found!r}   [{status}]")


run_tests("regex 1 (repeated words)", regex_1, re.IGNORECASE, [
    ("Humbert Humbert is a character character", True),
    ("the the", True),
    ("The the cat", True),
    ("the bug", False),
    ("the big bug", False),
    ("the theory", False),       # prefix of the next word is NOT a repetition
])

run_tests("regex 2 (integer ... word)", regex_2, re.MULTILINE, [
    ("42 is the answer to everything", True),
    ("42 hello", True),
    ("The answer is 42", False),   # does not start with an integer
    ("42 is the answer 42", False),  # ends with a number, not a word
    ("42 is the answer.", False),  # ends with punctuation
    ("42", False),
    ("3.14 is pi", False),         # 3.14 is not an integer
])

run_tests("regex 3 (grotto + raven)", regex_3, re.IGNORECASE | re.DOTALL, [
    ("The raven visited the grotto", True),
    ("grotto raven", True),
    ("The grotto\nand the Raven", True),
    ("The raven visited the grottos", False),
    ("Only a raven", False),
    ("Only a grotto", False),
])

run_tests("regex 4 (first word + punctuation)", regex_4, 0, [
    ("Hello, world! How are you?", True),
    ("She was pleased.", True),
    ("Don't stop.", True),
    ("Wait... what?", True),
    ("  Yes!", True),
    ("42 is the answer", False),
])


Testing regex 1 (repeated words): \b([A-Za-z]+)\b[\W_]+\b\1\b
  'Humbert Humbert is a character character'              -> 'Humbert Humbert'   [OK]
  'the the'                                               -> 'the the'   [OK]
  'The the cat'                                           -> 'The the'   [OK]
  'the bug'                                               -> None   [OK]
  'the big bug'                                           -> None   [OK]
  'the theory'                                            -> None   [OK]

Testing regex 2 (integer ... word): ^\d+\s+.*[A-Za-z]+$
  '42 is the answer to everything'                        -> '42 is the answer to everything'   [OK]
  '42 hello'                                              -> '42 hello'   [OK]
  'The answer is 42'                                      -> None   [OK]
  '42 is the answer 42'                                   -> None   [OK]
  '42 is the answer.'                                     -> None   [OK]
  '42'              

## Exercise 3: Regular Expression for Employee Data Extraction

Using the provided HTML file containing employee information, write regular expressions to extract the following data: Full Name, Email Address, Phone Number, Personal Identification Code, Date of Birth, Website URL, City, Country, Job Title, Department.

Create a regex for each data point and save the extracted employee data in a JSON file.

In [36]:
import re
import json
import html

# Read the HTML content from the file
with open("employee-data-html.html", "r", encoding="utf-8") as f:
    html_content = f.read()

# Regex patterns, written in the same style as Exercise 2.
regex_department = r'<h2>\s*(.*?)\s*</h2>'
regex_job_title = r'<h3>\s*(.*?)\s*</h3>'
regex_first_name = r'<span class="firstname">\s*(.*?)\s*</span>'
regex_last_name = r'<span class="lastname">\s*(.*?)\s*</span>'
regex_email = r'Email:\s*([^<\s]+)'
regex_phone = r'Téléphone:\s*([^<]+)'
regex_id_code = r"Code d'identification:\s*([^<]+)"
regex_birth_date = r'Date de naissance:\s*(\d{4}-\d{2}-\d{2})'
regex_website = r'Site Web:\s*(https?://[^\s<]+|www\.[^\s<]+)'
regex_location = r'Ville:\s*([^<]+)'
regex_section = r'<div class="department-section"[^>]*>(.*?)</div>\s*</div>\s*</div>'
regex_employee = r'<div class="employee-card">(.*?)</div>'


def get_match(pattern, text):
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    return html.unescape(match.group(1).strip()) if match else None


# Extract each department, then each employee card inside that department.
employees = []
sections = re.findall(regex_section, html_content, re.DOTALL)

for section in sections:
    department = get_match(regex_department, section)
    employee_cards = re.findall(regex_employee, section, re.DOTALL)

    for card in employee_cards:
        first_name = get_match(regex_first_name, card)
        last_name = get_match(regex_last_name, card)
        location = get_match(regex_location, card)
        city, country = location.split(',', 1) if location and ',' in location else (location, None)

        employees.append({
            'full_name': f'{first_name} {last_name}',
            'email': get_match(regex_email, card),
            'phone': get_match(regex_phone, card),
            'identification_code': get_match(regex_id_code, card),
            'date_of_birth': get_match(regex_birth_date, card),
            'website': get_match(regex_website, card),
            'city': city.strip() if city else None,
            'country': country.strip() if country else None,
            'job_title': get_match(regex_job_title, card),
            'department': department,
        })

# Save the extracted data in JSON format.
with open('employee_data.json', 'w', encoding='utf-8') as f:
    json.dump(employees, f, ensure_ascii=False, indent=2)

print(f'Extracted {len(employees)} employees and saved them to employee_data.json')
print(json.dumps(employees[0], ensure_ascii=False, indent=2))

Extracted 9 employees and saved them to employee_data.json
{
  "full_name": "Karim Bendaoud",
  "email": "k.bendaoud@techmaghreb.dz",
  "phone": "+213 551 23 45 67",
  "identification_code": "BD93#k4R1m*25",
  "date_of_birth": "1993-05-25",
  "website": "https://karim-bendaoud.dz",
  "city": "Alger",
  "country": "Algérie",
  "job_title": "Ingénieur Logiciel",
  "department": "Département d'Ingénierie"
}


## Exercise 4: Create a Chatbot Inspired by ELIZA

Design a simple chatbot that mimics the style of ELIZA, an early AI program created in 1966 by Joseph Weizenbaum. ELIZA simulated a conversation with a psychotherapist by using pattern matching and simple responses.

In [1]:
import re

# Enhanced ELIZA-like chatbot
class ElizaChatbot:
    def __init__(self):
        self.response_rules = [
            (r'\bquit\b', 'Goodbye! It was nice talking to you.'),
            (r'\b(hello|hi)\b', 'Hello there! How are you feeling today?'),
            (r'i need (.*)', r'Why do you need \1?'),
        ]

    def eliza_response(self, user_input):
        user_input = user_input.strip()

        # Check for patterns with case insensitivity
        for pattern, response in self.response_rules:
            match = re.search(pattern, user_input, re.IGNORECASE)
            if match:
                # Use re.sub to inject capture groups (like \1) into the response
                return re.sub(pattern, response, user_input, flags=re.IGNORECASE)

        # Return a default response if no patterns matched
        return "I didn't quite understand that. Can you elaborate?"

# Example usage
if __name__ == "__main__":
    eliza = ElizaChatbot()
    print("ELIZA: Hello! I'm your virtual assistant. Type 'quit' to exit.")

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("ELIZA: Goodbye! It was nice talking to you.")
            break
        print("ELIZA:", eliza.eliza_response(user_input))

ELIZA: Hello! I'm your virtual assistant. Type 'quit' to exit.
ELIZA: Hello there! How are you feeling today?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: I didn't quite understand that. Can you elaborate?
ELIZA: Goodbye! It was nice talking to you.


## Exercise 5: Edit Distance

1. Compute the edit distance (using insertion cost 1, deletion cost 1, substitution cost 1) of "leda" to "deal". Show your work (using the edit distance grid).

2. Implement a minimum edit distance algorithm and use your hand-computed results to check your code.

In [3]:
# Function to build the minimum edit distance grid
def edit_distance_grid(s1, s2):
    m, n = len(s1), len(s2)

    # Create a 2D array to store distances
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Initialize the first row and column
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    #your code here
    for i in range(1, m + 1):
        for j in range(1, n + 1):
           dp[i][j] = min(
                dp[i - 1][j] + 1,  # Deletion
                dp[i][j - 1] + 1,  # Insertion
                dp[i - 1][j - 1] + (0 if s1[i - 1] == s2[j - 1] else 1)  # Substitution
            )

    return dp

# Function to calculate the minimum edit distance between two strings
def edit_distance(s1, s2):
    return edit_distance_grid(s1, s2)[-1][-1]

# Function to print the edit distance grid
def print_edit_distance_grid(s1, s2, dp):
    print(f"Edit distance grid for '{s1}' to '{s2}':")
    print("    " + "  ".join(["#"] + list(s2)))
    for i, row in enumerate(dp):
        row_label = "#" if i == 0 else s1[i - 1]
        print(f"{row_label:>2}  " + "  ".join(str(value) for value in row))

# Compute and display the edit distance for "leda" to "deal"
source, target = "leda", "deal"

edit_distance_value = edit_distance(source, target)
print(f"Edit distance between '{source}' and '{target}': {edit_distance_value}")
print_edit_distance_grid(source, target, edit_distance_grid(source, target))




Edit distance between 'leda' and 'deal': 3
Edit distance grid for 'leda' to 'deal':
    #  d  e  a  l
 #  0  1  2  3  4
 l  1  1  2  3  3
 e  2  2  1  2  3
 d  3  2  2  2  3
 a  4  3  3  2  3


# Exercise 6: Spell Correction using NLTK

In this exercise, we'll implement spell correction functions using NLTK and the concepts of edit distance.

## Question 1:
Write a function that takes a misspelled word and returns the closest valid word from a given textual corpus using edit distance.

**Hint:** You can use the top 5000 most frequent words from an NLTK Gutenberg text such as 'austen-emma.txt'.

## Question 2:
Use the function you created in Question 1 to correct spelling errors in full sentences.

**Hint:** You can randomly introduce spelling errors into existing sentences from the Gutenberg Corpus available in NLTK by altering long words (those with more than 5 characters) through the insertion of a random character.


## Question 3:
Implement the following steps to evaluate the effectiveness of your spell correction:

1. **Select Sentences**: Choose a set of sentences from the Gutenberg Corpus available in NLTK.
2. **Introduce Spelling Errors**: Randomly alter some long words in each sentence to create spelling errors (through the insertion of a random character). Ensure that only words longer than 5 characters as before.
3. **Correct the Sentences**: Use the function from Question 1 to correct the misspelled words in each altered sentence.
4. **Print Results**: For each original sentence, print the altered sentence and the suggested corrections for the misspelled words.
5. **Calculate Precision**: Evaluate the accuracy of the corrections by calculating the precision of the spell correction method on the provided test set in the code section below.


In [5]:
import nltk
from nltk.corpus import gutenberg
from nltk import FreqDist, edit_distance
from nltk.tokenize import word_tokenize
import random
from sklearn.metrics import precision_score
import string

# Step 1: Download necessary NLTK data
nltk.download('punkt')
nltk.download('gutenberg')
nltk.download('words')

# Define the vocab size
Vocab_size = 5000

# Step 2: Load text and tokenize, filtering punctuation and converting to lowercase
tokens = [word.lower() for word in word_tokenize(gutenberg.raw('austen-emma.txt')) if word.isalpha()]
vocab = [word for word, _ in FreqDist(tokens).most_common(Vocab_size)]

#your own code here ....
def closest_word(word, vocab):
    """Find the closest word in vocab to the given word based on edit distance."""
    return min(vocab, key=lambda candidate: edit_distance(word, candidate))



# Test on a set of sentences from 'austen-emma.txt'
test_sentences = [
    "She was pleased to have such a friend.",
    "He thought it was a great opportunity.",
    "The sun was shining brightly in the sky.",
    "They were walking towards the market together.",
    "I hope you will come to the party."
]

# Question 2: Correct spelling errors in full sentences
def introduce_errors(sentence):
    """Randomly alter words longer than 5 characters by inserting a character."""
    words = word_tokenize(sentence)
    altered = []
    for w in words:
        if len(w) > 5 and w.isalpha():
            i = random.randint(0, len(w))
            w = w[:i] + random.choice(string.ascii_lowercase) + w[i:]
        altered.append(w)
    return " ".join(altered)

def correct_sentence(sentence, vocab):
    # shortcut: punctuation detokenization uses ' '.join, use TreebankWordDetokenizer for cleaner spacing
    words = word_tokenize(sentence)
    vocab_set = set(vocab)
    corrected = []
    for w in words:
        if w.isalpha() and w.lower() not in vocab_set:
            cand = closest_word(w.lower(), vocab)
            corrected.append(cand.capitalize() if w.istitle() else cand)
        else:
            corrected.append(w)
    return " ".join(corrected)

# Demonstration on test sentences
random.seed(42)
for sent in test_sentences:
    corrupted = introduce_errors(sent)
    print("Original: ", sent)
    print("Altered:  ", corrupted)
    print("Corrected:", correct_sentence(corrupted, vocab))
    print()

[nltk_data] Downloading package punkt to /Users/skout/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package gutenberg to /Users/skout/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package words to /Users/skout/nltk_data...
[nltk_data]   Package words is already up-to-date!


Original:  She was pleased to have such a friend.
Altered:   She was paleased to have such a frienid .
Corrected: She was pleased to have such a friend .

Original:  He thought it was a great opportunity.
Altered:   He thohught it was a great opxportunity .
Corrected: He thought it was a great opportunity .

Original:  The sun was shining brightly in the sky.
Altered:   The sun was svhining brightlyc in the sky .
Corrected: The sun was evening rightly in the sky .

Original:  They were walking towards the market together.
Altered:   They were walkinbg ctowards the mharket togethert .
Corrected: They were walking towards the market together .

Original:  I hope you will come to the party.
Altered:   I hope you will come to the party .
Corrected: I hope you will come to the party .



# Exercise 7: Implementing Byte Pair Encoding (BPE) Tokenization

In this exercise, you will implement the Byte Pair Encoding (BPE) tokenization method step by step. Follow the instructions below to build your BPE tokenizer.

### Step 1: Get Vocabulary

Implement the function `get_vocab(sentences)` that takes a list of sentences and returns a dictionary with the counts of each word in the corpus, where each word is written as space-separated characters followed by the end-of-word marker `</w>`.

### Step 2: Get Stats

Implement the function `get_stats(vocab)` that takes the vocabulary dictionary and returns a dictionary of pairs of tokens and their frequencies.

### Step 3: Merge Vocabulary

Implement the function `merge_vocab(vocab, pair)` that takes the vocabulary and a pair of tokens to merge. It should return the updated vocabulary.

### Step 4: BPE

Implement the function `bpe(sentences, num_merges)` that uses the previously defined functions to learn the BPE merges on the provided sentences.

### Step 5: Apply BPE

Implement the function `apply_bpe(sentence, merges)` that takes a sentence and the list of learned merges and returns the tokenized version of the sentence using BPE.

Once you have completed all steps, test your implementation with sample sentences to observe how the BPE tokenization works. You can adjust the number of merges to see how it affects the tokenization.


In [40]:
from collections import defaultdict
import nltk
nltk.download('punkt', quiet=True)      # older NLTK versions
nltk.download('punkt_tab', quiet=True)  # recent NLTK versions


# Step 1: Get Vocabulary
def get_vocab(sentences):
    """
    Functionality:
    - Processes a list of sentences and builds a vocabulary where each word is split into
      individual characters, followed by the end-of-word marker '</w>'.
    - '</w>' helps the algorithm distinguish the end of a word from the beginning of the next
      one. It is crucial for keeping word boundaries during the merging process.
    - The vocabulary is a dictionary: key = the word written as space-separated tokens,
      value = how many times the word appears in the corpus.

    Input:
    - sentences: list of strings (sentences or words).
          shape: list of length N, e.g. ["low", "lowest"]

    Output:
    - vocab: dict {str: int}
          shape: one entry per UNIQUE word in the corpus.

    Example:
        sentences = ["low", "lowest"]
        output    = {"l o w </w>": 1, "l o w e s t </w>": 1}

    Hints:
    - Use a defaultdict(int) so you can do vocab[key] += 1 without checking if the key exists.
    - For each sentence: lowercase it, then split it into words with nltk.word_tokenize(...).
    - For each word: list(word) gives its characters; add '</w>' as a last token.
    - Join the tokens with ' ' to build the key, then count it.
    - Return a normal dict (dict(vocab)).
    """
    vocab = defaultdict(int)

    # TODO: loop over the sentences
    for sentence in sentences:
    #   TODO: lowercase the sentence and tokenize it into words with nltk.word_tokenize
        sentence = sentence.lower() 
        words = nltk.word_tokenize(sentence)
    #   TODO: loop over the words
        for word in words:
    #       TODO: build the list of tokens: the characters of the word + ['</w>']
            tokens = list(word) + ['</w>']
    #       TODO: join the tokens with a space and increment the count in vocab
            key = ' '.join(tokens)
            vocab[key] += 1
    pass

    return dict(vocab)


# Step 2: Get Stats
def get_stats(vocab):
    """
    Functionality:
    - Computes the frequency of every pair of ADJACENT tokens (bigram) in the vocabulary.
    - Each pair is counted once per occurrence, weighted by the frequency of the word it
      appears in.
    - '</w>' is treated like any other token.

    Input:
    - vocab: dict {str: int}, keys are space-separated tokens, values are word frequencies.

    Output:
    - pairs: dict {(str, str): int}
          keys   = tuples of two str
          values = total frequency of that pair across the whole vocabulary

    Example:
        vocab = {"l o w </w>": 1, "l o w e s t </w>": 1}
        pairs = {('l','o'): 2, ('o','w'): 2, ('w','</w>'): 1,
                 ('w','e'): 1, ('e','s'): 1, ('s','t'): 1, ('t','</w>'): 1}

    Hints:
    - A word with k tokens has k - 1 adjacent pairs.
    - Use word.split() to get the list of tokens back from the key.
    - Add `freq` (not 1) to the pair count.
    """
    pairs = defaultdict(int)

    # TODO: loop over (word, freq) in vocab.items()
    for word, freq in vocab.items():
    #   TODO: split the word into tokens
        tokens = word.split()
    #   TODO: loop over the positions i from 0 to len(tokens) - 2
        for i in range(len(tokens) - 1):
    #       TODO: add freq to pairs[(tokens[i], tokens[i + 1])]
            pairs[(tokens[i], tokens[i + 1])] += freq
    pass

    return pairs


# Helper (PROVIDED, do not modify)
def merge_tokens(tokens, pair):
    """
    Merges every adjacent occurrence of `pair` in a LIST of tokens (left to right).

    Input:
    - tokens: list of str, e.g. ['l', 'o', 'w', '</w>']
    - pair:   tuple of two str, e.g. ('o', 'w')

    Output:
    - list of str, e.g. ['l', 'ow', '</w>']

    Why work on a list and not on the raw string?
    With str.replace, the pair ('a', 'b') would wrongly turn "ba b" into "bab".
    Working on tokens avoids merging across token boundaries.
    """
    merged, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged


# Step 3: Merge Vocabulary
def merge_vocab(vocab, pair):
    """
    Functionality:
    - Merges the given pair of tokens into ONE single token in every word of the vocabulary
      and returns the updated vocabulary.

    Input:
    - vocab: dict {str: int}, keys are space-separated tokens, values are word frequencies.
    - pair:  tuple (str, str), the two tokens to merge, e.g. ('o', 'w').

    Output:
    - new_vocab: dict {str: int}, same words but with the pair merged.

    Example:
        old vocab = {"l o w </w>": 1, "l o w e s t </w>": 1}
        pair      = ('o', 'w')
        new vocab = {"l ow </w>": 1, "l ow e s t </w>": 1}

    Hints:
    - Use the provided helper merge_tokens(tokens, pair) on the list word.split().
    - Join the result back with ' ' to obtain the new key.
    - Two different words can become identical after a merge, so ADD the frequencies:
      new_vocab[new_word] = new_vocab.get(new_word, 0) + freq
    """
    new_vocab = {}

    # TODO: loop over (word, freq) in vocab.items()
    for word, freq in vocab.items():
    #   TODO: split the word into tokens and merge the pair with merge_tokens
        tokens = word.split()
        merged_tokens = merge_tokens(tokens, pair)
    #   TODO: join the merged tokens with a space to get new_word
        new_word = ' '.join(merged_tokens)
    #   TODO: add freq to new_vocab[new_word]
        new_vocab[new_word] = new_vocab.get(new_word, 0) + freq
    pass

    return new_vocab


# Step 4: BPE (Byte Pair Encoding)
def bpe(sentences, num_merges):
    """
    Functionality:
    - Learns BPE merges: repeatedly finds the most frequent pair of tokens and merges it,
      num_merges times (or until there is nothing left to merge).

    Input:
    - sentences:  list of str, the training corpus.
    - num_merges: int, the number of merges to perform.

    Output:
    - vocab:  dict {str: int}, the vocabulary after all the merges.
    - merges: list of tuples (str, str), the merged pairs IN THE ORDER they were learned.
              len(merges) <= num_merges.

    Hints:
    - Start from get_vocab(sentences) and an empty list `merges`.
    - At each iteration: pairs = get_stats(vocab).
    - If pairs is empty, stop (break).
    - best_pair = max(pairs, key=pairs.get) gives the most frequent pair.
    - Update vocab with merge_vocab and store best_pair in merges.
    - The ORDER of merges matters: apply_bpe will need it.
    """
    # TODO: initialize vocab with get_vocab
    vocab = get_vocab(sentences)
    merges = []

    # TODO: repeat num_merges times:
    for _ in range(num_merges):

    #   TODO: compute the pair statistics
        pairs = get_stats(vocab)
        #   TODO: stop if there are no pairs
        if not pairs:
            break
    #   TODO: find the most frequent pair
        best_pair = max(pairs, key=pairs.get)
    #   TODO: merge it in the vocabulary
        vocab = merge_vocab(vocab, best_pair)
    #   TODO: record it in merges
        merges.append(best_pair)
    pass

    return vocab, merges


# Step 5: Apply BPE
def apply_bpe(sentence, merges):
    """
    Functionality:
    - Tokenizes a NEW sentence using the learned merges, applying each merge in the order in
      which it was learned.

    Input:
    - sentence: str, e.g. "lowering in the fest"
    - merges:   list of tuples (str, str), as returned by bpe().

    Output:
    - list of str: the BPE tokens of the sentence (the '</w>' marker stays in the tokens).

    Hints:
    - Use the SAME pre-processing as in training: lowercase + nltk.word_tokenize.
    - For each word: start from list(word) + ['</w>'].
    - Then, for each pair in merges (in order), tokens = merge_tokens(tokens, pair).
    - Collect the tokens of all the words in a single flat list (extend, not append).
    """
    tokenized_words = []

    # TODO: lowercase the sentence and split it into words with nltk.word_tokenize
    sentence = sentence.lower()
    words = nltk.word_tokenize(sentence)
    # TODO: for each word:
    for word in words:
    #   TODO: start with its characters + ['</w>']
        tokens = list(word) + ['</w>']
    #   TODO: apply every merge, in the learned order, with merge_tokens
        for pair in merges:
            tokens = merge_tokens(tokens, pair)
    #   TODO: add the resulting tokens to tokenized_words
        tokenized_words.extend(tokens)  
    pass

    return tokenized_words


# Example Usage / Tests
if __name__ == "__main__":
    # Quick checks: uncomment each one after finishing the corresponding step
    # assert get_vocab(["low", "lowest"]) == {"l o w </w>": 1, "l o w e s t </w>": 1}
    # assert get_stats({"l o w </w>": 1, "l o w e s t </w>": 1})[('o', 'w')] == 2
    # assert merge_vocab({"l o w </w>": 1, "l o w e s t </w>": 1}, ("o", "w")) == \
    #     {"l ow </w>": 1, "l ow e s t </w>": 1}
    # assert merge_vocab({"ba b </w>": 1}, ("a", "b")) == {"ba b </w>": 1}

    # Sample sentences for BPE training
    sentences = [
        "Low",
        "lowest",
        "the",
        "this",
        "fastest"
    ]

    # Perform BPE with a specified number of merges (try different values!)
    num_merges = 10
    vocab, merges = bpe(sentences, num_merges)

    print("Vocabulary after BPE:", vocab)
    print("Learned merges:", merges)

    # Apply BPE to a new sentence that wasn't part of the training sentences
    sentence = "lowering in the fest"
    tokenized_sentence = apply_bpe(sentence, merges)

    print("Tokenized sentence:", tokenized_sentence)

Vocabulary after BPE: {'low</w>': 1, 'lowest</w>': 1, 'the</w>': 1, 'th i s </w>': 1, 'f a st est</w>': 1}
Learned merges: [('s', 't'), ('l', 'o'), ('lo', 'w'), ('e', 'st'), ('est', '</w>'), ('t', 'h'), ('low', '</w>'), ('low', 'est</w>'), ('th', 'e'), ('the', '</w>')]
Tokenized sentence: ['low', 'e', 'r', 'i', 'n', 'g', '</w>', 'i', 'n', '</w>', 'the</w>', 'f', 'est</w>']


# Exercise 8: Using SentencePiece for Byte Pair Encoding (BPE) Tokenization

In this exercise, you will learn how to use the SentencePiece library to implement Byte Pair Encoding (BPE) for tokenization. Follow the steps below to tokenize text using BPE.

### Step 1: Install SentencePiece

Ensure that you have the SentencePiece library installed in your Python environment. You can install it using pip:

```python
!pip install sentencepiece
```

### Step 2: Prepare Your Text
Load and merge all texts from the Gutenberg corpus available in NLTK. You can use the following code snippet to achieve this:

### Step 3: Train the BPE Model
Utilize SentencePiece to train a BPE model on the merged text. Specify the vocabulary size you want to achieve during the training process. **Important:** the default model type of SentencePiece is `unigram`, so you must ask explicitly for `model_type='bpe'`. For example:
```python
import sentencepiece as spm

# Train the SentencePiece model
spm.SentencePieceTrainer.train(input='merged_gutenberg.txt', model_prefix='m',
                               vocab_size=5000, model_type='bpe')
```


### Step 4: Tokenize the Text
Once the model is trained, use it to tokenize your text into subword units. You can test the tokenizer with the following sample sentences:

```python
# Load the trained model
sp = spm.SentencePieceProcessor(model_file='m.model')

# Tokenize a sentence
tokens = sp.encode(sentence, out_type=str)
```

### Step 5: What about Arabic Texts !
Try to do the same thing but this time using an Arabic text corpus.

In [41]:
# Step 1: Install SentencePiece
import sentencepiece as spm

# Step 2: Prepare Your Text
import nltk
from nltk.corpus import gutenberg

# Download the Gutenberg corpus if not already downloaded
nltk.download('gutenberg')

# Get the list of file IDs from the Gutenberg corpus
file_ids = gutenberg.fileids()

# Merge all texts into a single string
merged_text = ""
for file_id in file_ids:
    merged_text += gutenberg.raw(file_id) + "\n"

# Write the merged text to a temporary file
with open('merged_gutenberg.txt', 'w', encoding='utf-8') as f:
    f.write(merged_text)

# Step 3: Train the BPE Model
spm.SentencePieceTrainer.train(input='merged_gutenberg.txt', model_prefix='m', vocab_size=5000, model_type='bpe')

# Step 4: Tokenize the Text
sp = spm.SentencePieceProcessor(model_file='m.model')
sentence = "lowering in the fest"
tokens = sp.encode(sentence, out_type=str)
print("Tokens:", tokens)

# Step 5: What about Arabic Texts !
from nltk.corpus import udhr
nltk.download('udhr')
with open('merged_arabic.txt', 'w', encoding='utf-8') as f:
    f.write(udhr.raw('Arabic_Alarabia-Arabic'))

spm.SentencePieceTrainer.train(input='merged_arabic.txt', model_prefix='m_ar', vocab_size=500, model_type='bpe')
sp_ar = spm.SentencePieceProcessor(model_file='m_ar.model')
sentence_ar = "الإعلان العالمي لحقوق الإنسان"
tokens_ar = sp_ar.encode(sentence_ar, out_type=str)
print("Arabic tokens:", tokens_ar)

# --- Verification Tests ---

# 1. Check exact vocabulary sizes match requested limits
assert sp.get_piece_size() == 5000, f"Expected 5000, got {sp.get_piece_size()}"
assert sp_ar.get_piece_size() == 500, f"Expected 500, got {sp_ar.get_piece_size()}"

# 2. Test Lossless Roundtrip (encode -> decode must return the exact string)
test_en = "SentencePiece tokenizes text losslessly!"
pieces_en = sp.encode(test_en, out_type=str)
assert sp.decode(pieces_en) == test_en
print("EN Pieces :", pieces_en)
print("EN Decoded:", sp.decode(pieces_en))

test_ar = "الإعلان العالمي لحقوق الإنسان"
pieces_ar = sp_ar.encode(test_ar, out_type=str)
assert sp_ar.decode(pieces_ar) == test_ar
print("AR Pieces :", pieces_ar)
print("AR Decoded:", sp_ar.decode(pieces_ar))

# 3. Test Numerical ID Encoding (used by neural networks)
ids = sp.encode(test_en, out_type=int)
print("Token IDs :", ids)
assert sp.decode(ids) == test_en

print("\nAll verification tests passed!")


[nltk_data] Downloading package gutenberg to /Users/skout/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
I0000 00:00:1792938203.108932 5959026 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: merged_gutenberg.txt
  input_format: 
  model_prefix: m
  model_type: BPE
  vocab_size: 5000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos

I0000 00:00:1792938203.325030 5959026 trainer_interface.cc:555] all chars count=11622512
I0000 00:00:1792938203.325075 5959026 trainer_interface.cc:566] Done: 99.9532% characters are covered.
I0000 00:00:1792938203.325079 5959026 trainer_interface.cc:578] Alphabet size=70
I0000 00:00:1792938203.325080 5959026 trainer_interface.cc:579] Final character coverage=0.999532


I0000 00:00:1792938203.555618 5959026 trainer_interface.cc:609] Done! preprocessed 206089 sentences.
I0000 00:00:1792938203.555668 5959026 trainer_interface.cc:615] Tokenizing input sentences with whitespace: 206089
I0000 00:00:1792938203.707129 5959026 trainer_interface.cc:626] Done! 124427


Tokens: ['▁low', 'ering', '▁in', '▁the', '▁f', 'est']
Arabic tokens: ['▁الإعلان', '▁العال', 'مي', '▁ل', 'حقوق', '▁الإنسان']
EN Pieces : ['▁S', 'ent', 'ence', 'P', 'ie', 'ce', '▁to', 'ken', 'iz', 'es', '▁te', 'xt', '▁loss', 'less', 'ly', '!']
EN Decoded: SentencePiece tokenizes text losslessly!
AR Pieces : ['▁الإعلان', '▁العال', 'مي', '▁ل', 'حقوق', '▁الإنسان']
AR Decoded: الإعلان العالمي لحقوق الإنسان
Token IDs : [100, 77, 394, 4988, 257, 81, 32, 1108, 821, 30, 779, 1171, 3823, 617, 78, 4981]

All verification tests passed!


I0000 00:00:1792938203.833911 5959026 bpe_model_trainer.cc:321] Added: freq=71337 size=20 all=2274 active=2202 piece=▁f
I0000 00:00:1792938203.852525 5959026 bpe_model_trainer.cc:321] Added: freq=39630 size=40 all=3305 active=3233 piece=at
I0000 00:00:1792938203.861489 5959026 bpe_model_trainer.cc:321] Added: freq=25651 size=60 all=4347 active=4275 piece=im
I0000 00:00:1792938203.870147 5959026 bpe_model_trainer.cc:321] Added: freq=18583 size=80 all=5426 active=5354 piece=▁not
I0000 00:00:1792938203.875837 5959026 bpe_model_trainer.cc:321] Added: freq=14738 size=100 all=6404 active=6332 piece=▁we
I0000 00:00:1792938203.879808 5959026 bpe_model_trainer.cc:321] Added: freq=11916 size=120 all=7360 active=7288 piece=ore
I0000 00:00:1792938203.883678 5959026 bpe_model_trainer.cc:321] Added: freq=9926 size=140 all=8226 active=8154 piece=▁thou
I0000 00:00:1792938203.888934 5959026 bpe_model_trainer.cc:321] Added: freq=8531 size=160 all=8970 active=8898 piece=▁C
I0000 00:00:1792938203.891490 5

## Main differences between our simple BPE implementation and SentencePiece BPE:

### 1. Word-boundary marker (end-of-word vs start-of-word):
- **Simple BPE:** Uses `</w>` at the end of each word.
- **SentencePiece:** Uses `▁` (U+2581 "lower one eighth block", NOT the ASCII underscore `_`) at the beginning of each word.

**Example:**
- Simple BPE: `"low</w> est</w>"`
- SentencePiece: `"▁low ▁est"`

---

### 2. Tokenization approach:
- **Simple BPE:** Pre-tokenizes the text into words (here with NLTK), then applies BPE inside each word.
- **SentencePiece:** Treats the input as a raw stream of characters, spaces included (no external tokenizer is needed, so it is language independent). By default the learned pieces do not cross word boundaries.

**Example (before any merge / after training):**
- Simple BPE: `["l", "o", "w", "</w>", "e", "s", "t", "</w>"]` -> `["low</w>", "est</w>"]`
- SentencePiece: `["▁", "l", "o", "w", "▁", "e", "s", "t"]` -> `["▁low", "▁est"]`

---

### 3. Handling of whitespace:
- **Simple BPE:** Whitespace is discarded during tokenization (only the `</w>` marker remains).
- **SentencePiece:** Whitespace is kept as part of the tokens (it is replaced by `▁`).

**Example:**
- Simple BPE: `"hello world"` -> `["hello</w>", "world</w>"]`
- SentencePiece: `"hello world"` -> `["▁hello", "▁world"]`

---

### 4. Subword regularization:
- **Simple BPE:** No built-in regularization (segmentation is deterministic).
- **SentencePiece:** Supports subword regularization: sampling from the unigram model and BPE-dropout for BPE models.

---

### 5. Unicode handling:
- **Simple BPE:** Python works on Unicode characters, but our pipeline relies on `nltk.word_tokenize`, which is designed for English-like text (it would need another tokenizer for Arabic, Chinese, Japanese...).
- **SentencePiece:** Language independent: works directly on Unicode characters (optionally on bytes with `byte_fallback`), with `character_coverage` controlling which rare characters are kept.

---

### 6. Vocabulary size control:
- **Simple BPE:** Controlled indirectly by the number of merge operations (final vocabulary = initial characters + merges).
- **SentencePiece:** The desired vocabulary size is specified directly (`vocab_size`, which also counts the special tokens `<unk>`, `<s>`, `</s>`).

---

### 7. Out-of-vocabulary (OOV) handling:
- **Simple BPE:** Any word can be split into known characters, but a character never seen in training is simply left as it is (no special token).
- **SentencePiece:** Characters outside the vocabulary are mapped to the special `<unk>` token, unless `byte_fallback` is enabled (bytes are then used instead).

---

### 8. Reversibility (lossless tokenization):
- **Simple BPE:** The output is not a faithful copy of the input: our pipeline lower-cases the text and `word_tokenize` changes quotes, splits punctuation and loses the original spacing, so the exact text cannot be recovered.
- **SentencePiece:** Lossless: whitespace is kept as `▁`, so detokenization is just concatenating the pieces and turning `▁` back into a space (no external detokenizer needed; only the built-in normalization can change the text: Unicode NFKC, and by default removal of leading/trailing/duplicate spaces).

**Example:**
- SentencePiece: `["▁app", "le"]` -> `"▁apple"` -> `"apple"`
- Simple BPE: `["app", "le</w>"]` -> `"apple</w>"` -> `"apple "` (case and spacing of the original text are already lost before BPE even starts)

---

### 9. Efficiency and speed:
- **Simple BPE:** Very slow on large datasets (pure Python, the whole vocabulary is rescanned after every merge).
- **SentencePiece:** Implemented in C++ and highly optimized for speed and efficiency.

### 10. Default model type:
- **SentencePiece** trains a **unigram** language model by default; BPE must be requested with `model_type='bpe'`.